<a href="https://colab.research.google.com/github/ZukaCS/ML_assignment4/blob/main/SmallCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle wandb onnx -Uq

In [2]:
import wandb
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!mkdir -p ~/.kaggle
!cp "/content/drive/MyDrive/ML_assignment4/kaggle.json" ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [4]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d data/
!tar -xzf data/fer2013.tar.gz -C data/

replace data/example_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace data/fer2013.tar.gz? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace data/icml_face_data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
y
replace data/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: replace data/train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
y


In [6]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: zkeke23 (zkeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms


In [8]:
train_df = pd.read_csv('data/fer2013/fer2013.csv')
train_df.head()

,emotion,pixels,Usage
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...,Training
1,0,151 150 147 155 148 133 111 140 170 174 182 15...,Training
2,2,231 212 156 164 174 138 161 173 182 200 106 38...,Training
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...,Training
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...,Training


In [9]:
train_df.shape

(35887, 3)

In [10]:
import random

def set_seed(seed=50):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(50)

In [11]:
class FERDataset(Dataset):
    def __init__(self, csv_file, split='Training', transform=None):
        self.data = pd.read_csv(csv_file)
        self.data = self.data[self.data['Usage'] == split]
        self.transform = transform
        self.images = self.data['pixels'].apply(lambda x: np.array(x.split(), dtype=np.float32).reshape(48, 48))
        self.images = np.stack(self.images.values)
        self.labels = self.data['emotion'].values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        image = np.expand_dims(image, axis=2).astype(np.uint8)
        if self.transform:
            image = self.transform(image)
        return image, label

In [12]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
print("Loading datasets...")
train_dataset = FERDataset('data/fer2013/fer2013.csv', split='Training', transform=transform)
val_dataset = FERDataset('data/fer2013/fer2013.csv', split='PublicTest', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")


Loading datasets...
Train size: 28709, Val size: 3589


In [13]:
class SmallCNN(nn.Module):
    def __init__(self):
        super(SmallCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2) # maxpool from 48x48 to24x24

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2) # maxpool from 24x24 to 12x12

        self.fc1 = nn.Linear(32 * 12 * 12, 7)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 12 * 12)
        x = self.fc1(x)
        return x


In [14]:
import torch.optim as optim
import wandb
import wandb.util
import copy
import sklearn
from torch.utils.data import DataLoader

PROJECT = 'fer-challenge'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_sizes_to_test = [32, 64]
learning_rates_to_test = [0.001, 0.005, 0.0005]
epochs = 15

best_overall_val_acc = 0.0

try:
    wandb.finish()
except:
    pass

print("..Hyperparameter Tuning Loop (Batch Size and LR)..")

for bs in batch_sizes_to_test:
    print(f"\n====")
    print(f"Setting up DataLoaders for Batch Size: {bs}")
    print(f"=====")

    train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=bs, shuffle=False)

    for lr in learning_rates_to_test:
        print(f"\n:)")
        print(f"Testing Learning Rate: {lr} with Batch Size: {bs}")
        print(f":)")

        config = {
            'architecture': 'SmallCNN',
            'dataset': 'FER-2013',
            'optimizer': 'Adam',
            'loss_function': 'CrossEntropyLoss',
            'learning_rate': lr,
            'batch_size': bs,
            'epochs': epochs,
            'input_size': '(1, 48, 48)',
            'num_classes': 7,
            'seed': 50
        }

        run = wandb.init(
            project=PROJECT,
            name=f"SmallCNN_LR_{lr}_BS_{bs}",
            id=wandb.util.generate_id(),
            config=config,
            reinit=True
        )

        model = SmallCNN().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        best_val_acc_this_run = 0.0

        for epoch in range(epochs):
            model.train()
            running_loss, correct, total = 0.0, 0, 0

            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

            train_loss = running_loss / len(train_loader)
            train_acc = 100. * correct / total

            model.eval()
            val_loss, val_correct, val_total = 0.0, 0, 0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    val_total += labels.size(0)
                    val_correct += predicted.eq(labels).sum().item()

            val_loss = val_loss / len(val_loader)
            val_acc = 100. * val_correct / val_total

            acc_gap = train_acc - val_acc

            print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Gap: {acc_gap:.2f}%")

            if val_acc > best_val_acc_this_run:
                best_val_acc_this_run = val_acc

            if val_acc > best_overall_val_acc:
                best_overall_val_acc = val_acc
                torch.save(model.state_dict(), "best_model_smallCNN.pth")

            wandb.log({
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "acc_gap": acc_gap,
                "epoch": epoch + 1
            })

        wandb.run.summary["best_val_acc"] = best_val_acc_this_run
        wandb.finish()

print("\ntuning complete! The best smallCNN model was saved")


..Hyperparameter Tuning Loop (Batch Size and LR)..

====
Setting up DataLoaders for Batch Size: 32
=====

:)
Testing Learning Rate: 0.001 with Batch Size: 32
:)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/15 | Train Acc: 38.72% | Val Acc: 43.86% | Gap: -5.13%
Epoch 2/15 | Train Acc: 47.22% | Val Acc: 48.23% | Gap: -1.01%
Epoch 3/15 | Train Acc: 51.56% | Val Acc: 49.65% | Gap: 1.91%
Epoch 4/15 | Train Acc: 54.55% | Val Acc: 51.02% | Gap: 3.53%
Epoch 5/15 | Train Acc: 57.23% | Val Acc: 50.85% | Gap: 6.38%
Epoch 6/15 | Train Acc: 59.10% | Val Acc: 51.35% | Gap: 7.75%
Epoch 7/15 | Train Acc: 60.85% | Val Acc: 51.30% | Gap: 9.56%
Epoch 8/15 | Train Acc: 62.27% | Val Acc: 51.27% | Gap: 11.00%
Epoch 9/15 | Train Acc: 63.70% | Val Acc: 50.04% | Gap: 13.66%
Epoch 10/15 | Train Acc: 65.22% | Val Acc: 50.63% | Gap: 14.60%
Epoch 11/15 | Train Acc: 65.92% | Val Acc: 49.29% | Gap: 16.63%
Epoch 12/15 | Train Acc: 66.83% | Val Acc: 51.04% | Gap: 15.78%
Epoch 13/15 | Train Acc: 67.67% | Val Acc: 49.96% | Gap: 17.71%
Epoch 14/15 | Train Acc: 68.39% | Val Acc: 49.79% | Gap: 18.60%
Epoch 15/15 | Train Acc: 69.29% | Val Acc: 50.18% | Gap: 19.11%


acc_gap,▁▂▃▄▄▅▅▆▆▇▇▇███
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▃▄▅▅▆▆▆▇▇▇▇███
train_loss,█▆▅▅▄▃▃▃▂▂▂▂▁▁▁
val_acc,▁▅▆█████▇▇▆█▇▇▇
val_loss,▅▃▂▁▁▂▃▃▄▄▅▆█▇█
acc_gap,19.11075
best_val_acc,51.35135
epoch,15
train_acc,69.29186
train_loss,0.84363



:)
Testing Learning Rate: 0.005 with Batch Size: 32
:)


...

acc_gap,▁▄▅▅▅▆▇▇▇▆██▇██
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▄▅▅▆▆▇▇▇▇▇████
train_loss,█▆▄▄▃▃▂▂▂▂▂▁▁▁▁
val_acc,▁▂▅▅▆▆▅▆▆█▅▆▇▇▇
val_loss,█▅▃▃▂▁▂▁▁▁▃▂▁▁▁
acc_gap,6.48027
best_val_acc,48.73224
epoch,15
train_acc,53.87509
train_loss,1.22789



:)
Testing Learning Rate: 0.0005 with Batch Size: 64
:)


Epoch 1/15 | Train Acc: 36.20% | Val Acc: 42.32% | Gap: -6.12%
Epoch 2/15 | Train Acc: 44.43% | Val Acc: 45.22% | Gap: -0.79%
Epoch 3/15 | Train Acc: 47.84% | Val Acc: 46.56% | Gap: 1.28%
Epoch 4/15 | Train Acc: 50.11% | Val Acc: 48.93% | Gap: 1.18%
Epoch 5/15 | Train Acc: 51.97% | Val Acc: 49.15% | Gap: 2.82%
Epoch 6/15 | Train Acc: 53.65% | Val Acc: 48.90% | Gap: 4.75%
Epoch 7/15 | Train Acc: 54.40% | Val Acc: 50.77% | Gap: 3.63%
Epoch 8/15 | Train Acc: 55.86% | Val Acc: 50.35% | Gap: 5.52%
Epoch 9/15 | Train Acc: 56.98% | Val Acc: 50.38% | Gap: 6.60%
Epoch 10/15 | Train Acc: 58.49% | Val Acc: 50.96% | Gap: 7.53%
Epoch 11/15 | Train Acc: 59.40% | Val Acc: 50.79% | Gap: 8.61%
Epoch 12/15 | Train Acc: 60.53% | Val Acc: 50.07% | Gap: 10.46%
Epoch 13/15 | Train Acc: 61.40% | Val Acc: 50.88% | Gap: 10.52%
Epoch 14/15 | Train Acc: 62.23% | Val Acc: 51.13% | Gap: 11.11%
Epoch 15/15 | Train Acc: 63.08% | Val Acc: 50.43% | Gap: 12.65%


acc_gap,▁▃▄▄▄▅▅▅▆▆▆▇▇▇█
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▃▄▅▅▆▆▆▆▇▇▇███
train_loss,█▆▅▅▄▄▃▃▃▂▂▂▂▁▁
val_acc,▁▃▄▆▆▆█▇▇██▇██▇
val_loss,█▅▃▂▂▂▁▁▁▁▁▂▁▂▂
acc_gap,12.64591
best_val_acc,51.12845
epoch,15
train_acc,63.07778
train_loss,1.01275



tuning complete! The best smallCNN model was saved


In [15]:
#log other metrics for best model + confusion matrix
print("\nGenerating final evaluation metrics for the best model...")
run = wandb.init(project=PROJECT, name="SmallCNN_Best_Model_Evaluation")

eval_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model.load_state_dict(torch.load("best_model_smallCNN.pth"))
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in eval_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

class_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
    probs=None,
    y_true=all_labels,
    preds=all_preds,
    class_names=class_names
)})

from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score
import numpy as np

precision, sensitivity, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

cm = confusion_matrix(all_labels, all_preds)
specificities = []
for i in range(len(class_names)):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)

    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    specificities.append(spec)

macro_specificity = np.mean(specificities)
best_accuracy = accuracy_score(all_labels, all_preds)

wandb.log({
    "best_model_accuracy": best_accuracy,
    "best_model_precision": precision,
    "best_model_sensitivity": sensitivity,
    "best_model_specificity": macro_specificity,
    "best_model_f1": f1
})

wandb.save("best_model_smallCNN.pth")
wandb.finish()
print("Confusion Matrix and all Metrics successfully logged to WandB!")


Generating final evaluation metrics for the best model...


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


best_model_accuracy,▁
best_model_f1,▁
best_model_precision,▁
best_model_sensitivity,▁
best_model_specificity,▁
best_model_accuracy,0.52856
best_model_f1,0.51732
best_model_precision,0.52235
best_model_sensitivity,0.52856
best_model_specificity,0.91829


Confusion Matrix and all Metrics successfully logged to WandB!
